Arithmetic Operations on Images 
===============================

Goal
----

-   Learn several arithmetic operations on images, like addition, subtraction, bitwise operations, and etc.
-   Learn these functions: **cv.add()**, **cv.addWeighted()**, etc.

Image Addition
--------------

You can add two images with the OpenCV function, cv.add(), or simply by the numpy operation
res = img1 + img2. Both images should be of same depth and type, or the second image can just be a
scalar value.

> **Note:** There is a difference between OpenCV addition and Numpy addition. OpenCV addition is a
saturated operation while Numpy addition is a modulo operation.

For example, consider the below sample:

In [ ]:
import cv2 as cv
import numpy as np

x = np.uint8([250])
y = np.uint8([10])

cv.add(x, y)

In [ ]:
x + y  # 250+10 = 260 % 256 = 4

This will be more visible when you add two images. Stick with OpenCV functions, because they will provide a better result.

Image Blending
--------------

This is also image addition, but different weights are given to images in order to give a feeling of
blending or transparency. Images are added as per the equation below:

$$g(x) = (1 - \alpha)f_{0}(x) + \alpha f_{1}(x)$$

By varying $\alpha$ from $0 \rightarrow 1$, you can perform a cool transition between one image to
another.

Here I took two images to blend together. The first image is given a weight of 0.7 and the second image
is given 0.3. cv.addWeighted() applies the following equation to the image:

$$dst = \alpha \cdot img1 + \beta \cdot img2 + \gamma$$

Here $\gamma$ is taken as zero.

In [ ]:
import ipywidgets as widgets
from ipycanvas import Canvas

img1 = cv.imread("../../assets/ml.png")
img2 = cv.imread("../../assets/opencv-logo.png")

# Resize img2 to match img1 dimensions for addWeighted
img2 = cv.resize(img2, (img1.shape[1], img1.shape[0]))

# Canvas for live display
canvas = Canvas(width=img1.shape[1], height=img1.shape[0])

# Sliders for α, β, γ in: dst = α·img1 + β·img2 + γ
alpha_sl = widgets.FloatSlider(
    value=0.7, min=0, max=1, step=0.05, description="α (img1)"
)
beta_sl = widgets.FloatSlider(
    value=0.3, min=0, max=1, step=0.05, description="β (img2)"
)
gamma_sl = widgets.IntSlider(value=0, min=0, max=255, description="γ")


def update(*_, _img1=img1, _img2=img2):
    dst = cv.addWeighted(_img1, alpha_sl.value, _img2, beta_sl.value, gamma_sl.value)
    canvas.put_image_data(dst, 0, 0)


for w in (alpha_sl, beta_sl, gamma_sl):
    w.observe(update, "value")
update()

display(widgets.VBox([widgets.HBox([alpha_sl, beta_sl, gamma_sl]), canvas]))

Bitwise Operations
------------------

This includes the bitwise AND, OR, NOT, and XOR operations. They will be highly useful while extracting
any part of the image (as we will see in coming chapters), defining and working with non-rectangular
ROI's, and etc. Below we will see an example of how to change a particular region of an image.

I want to put the OpenCV logo above an image. If I add two images, it will change the color. If I blend them,
I get a transparent effect. But I want it to be opaque. If it was a rectangular region, I could use
ROI as we did in the last chapter. But the OpenCV logo is a not a rectangular shape. So you can do it with
bitwise operations as shown below:

In [ ]:
import ipywidgets as widgets
from ipycanvas import Canvas

# Create a synthetic background — bilinear gradient with black and white corners
W, H = 600, 400
xx, yy = np.meshgrid(np.linspace(0, 1, W), np.linspace(0, 1, H))
# top-left: black  ·  top-right: blue  ·  bot-left: green  ·  bot-right: white
b = (xx * 255).astype(np.uint8)  # 0 on left, 255 on right
g = (yy * 255).astype(np.uint8)  # 0 on top,  255 on bottom
r = (xx * yy * 255).astype(np.uint8)  # 0 along top & left edges, 255 at bottom-right
img1 = cv.merge([b, g, r])

# Load logo
img2 = cv.imread("../../assets/opencv-logo-white.png")
assert img2 is not None, "file could not be read, check with os.path.exists()"

# Pre-compute logo mask and foreground
logo_h, logo_w = img2.shape[:2]
img2gray = cv.cvtColor(img2, cv.COLOR_BGR2GRAY)
ret, mask = cv.threshold(img2gray, 10, 255, cv.THRESH_BINARY)
mask_inv = cv.bitwise_not(mask)
img2_fg = cv.bitwise_and(img2, img2, mask=mask)

# Canvas for live display
canvas = Canvas(width=W, height=H)

# Sliders to move the logo
x_sl = widgets.IntSlider(value=0, min=0, max=W - logo_w, description="X")
y_sl = widgets.IntSlider(value=0, min=0, max=H - logo_h, description="Y")

# Dropdown for the bitwise operation applied directly on the overlapping ROI
op_dd = widgets.Dropdown(
    options=[
        ("cv.add", "add"),
        ("cv.bitwise_and", "and"),
        ("cv.bitwise_or", "or"),
        ("cv.bitwise_xor", "xor"),
    ],
    value="add",
    description="Operation:",
)

ops = {
    "add": cv.add,
    "and": cv.bitwise_and,
    "or": cv.bitwise_or,
    "xor": cv.bitwise_xor,
}


def update(
    *_,
    _img1=img1,
    _img2_fg=img2_fg,
    _mask=mask,
    _mask_inv=mask_inv,
    _logo_h=logo_h,
    _logo_w=logo_w,
):
    out = _img1.copy()
    x, y = x_sl.value, y_sl.value
    # Guard against out-of-bounds slider values (can happen during widget state restore)
    x = max(0, min(x, _img1.shape[1] - _logo_w))
    y = max(0, min(y, _img1.shape[0] - _logo_h))
    roi = out[y : y + _logo_h, x : x + _logo_w]
    # Apply the bitwise operation directly between the ROI and the logo,
    # masked so only the logo shape is affected (outside the logo stays unchanged).
    result = ops[op_dd.value](roi, _img2_fg)
    blended = cv.bitwise_and(result, result, mask=_mask)  # affected area
    bg = cv.bitwise_and(roi, roi, mask=_mask_inv)  # unchanged area
    out[y : y + _logo_h, x : x + _logo_w] = cv.add(blended, bg)
    canvas.put_image_data(out, 0, 0)


for w in (x_sl, y_sl, op_dd):
    w.observe(update, "value")
update()

display(widgets.VBox([widgets.HBox([x_sl, y_sl, op_dd]), canvas]))

Exercises
---------

1. Create a slide show of images in a folder with smooth transition between images using
    cv.addWeighted function